In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from sklearn.preprocessing import StandardScaler

# ================================
# 📁 Paths
# ================================
save_path = "data/processed"
factor_path = os.path.join(save_path, "factor_library.parquet")
adj_path = os.path.join(save_path, "graph_adj.npy")

# ================================
# 📊 Load and Prepare Data
# ================================
factors = pd.read_parquet(factor_path)
print(f"✅ Loaded factor data: {factors.shape}")

adj = np.load(adj_path)
print(f"✅ Loaded adjacency matrix: {adj.shape}")

# Aggregate returns per ticker
tickers = factors['ticker'].unique()
features = []
for t in tickers:
    ret_series = factors.loc[factors['ticker'] == t, 'ret'].values
    features.append(ret_series)

# Pad features to equal length
max_len = max(len(f) for f in features)
features_padded = np.array([
    np.pad(f, (0, max_len - len(f)), 'constant') for f in features
])

# Handle NaNs and standardize
scaler = StandardScaler(with_mean=False)
X = scaler.fit_transform(np.nan_to_num(features_padded))

# Convert to torch tensor
X = torch.tensor(X, dtype=torch.float)

# Convert adjacency to edge_index format
edge_index = torch.tensor(np.array(np.nonzero(adj > 0)), dtype=torch.long)

# Create graph data object
data = Data(x=X, edge_index=edge_index)

# ================================
# 🧠 Define GNN Autoencoder Model
# ================================
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, latent_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, dropout=0.2)
        self.conv2 = GATv2Conv(hidden_channels * 4, latent_channels, heads=1, concat=False, dropout=0.2)
        self.decoder = torch.nn.Linear(latent_channels, in_channels)  # reconstruct original features

    def forward(self, x, edge_index):
        z = self.conv1(x, edge_index)
        z = F.elu(z)
        z = self.conv2(z, edge_index)
        x_hat = self.decoder(z)
        return z, x_hat  # return both latent and reconstructed

# ================================
# ⚙️ Training Setup
# ================================
in_channels = X.shape[1]
hidden_channels = 32
latent_channels = 16
lr = 0.005
epochs = 200

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN(in_channels, hidden_channels, latent_channels).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# ================================
# 🚀 Training Loop
# ================================
def train():
    model.train()
    optimizer.zero_grad()
    z, x_hat = model(data.x, data.edge_index)
    loss = F.mse_loss(x_hat, data.x)
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(1, epochs + 1):
    loss = train()
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{epochs}, Loss: {loss:.6f}")

# ================================
# 💾 Save Results
# ================================
model.eval()
with torch.no_grad():
    z, _ = model(data.x, data.edge_index)
    embeddings = z.cpu().numpy()

np.save(os.path.join(save_path, "node_embeddings.npy"), embeddings)
print("✅ Saved node embeddings to node_embeddings.npy")

alpha_predictions = pd.DataFrame({
    'ticker': tickers,
    'alpha': embeddings[:, 0]
})
alpha_predictions.to_parquet(os.path.join(save_path, "alpha_predictions.parquet"))
print("✅ Saved alpha predictions to alpha_predictions.parquet")

print("🎉 Training and saving complete.")
